In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
%pip install pyspark

In [41]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

# Create and config Spark
config = SparkConf().setAppName("bai04").setMaster("local[*]")
sparkContext = SparkContext.getOrCreate(conf=config)
sparkSession = SparkSession.builder.appName("bai04").getOrCreate()

In [33]:
data_path = "/content/drive/MyDrive/data/"

# Read file
movies_data = sparkContext.textFile(data_path + "movies.txt")

ratings1_data = sparkContext.textFile(data_path + "ratings_1.txt")
ratings2_data = sparkContext.textFile(data_path + "ratings_2.txt")

users_data = sparkContext.textFile(data_path + "users.txt")

In [42]:
# Parse users.txt
def parse_user_age(line):
    parts = line.split(',')
    user_id = int(parts[0])
    gender = parts[1]
    age = int(parts[2])
    occupation = int(parts[3])
    zipcode = parts[4]
    return (user_id, age)

# seperage age
def sperage_age_group(age):
    if age <= 18:
        return "0-18"
    elif age <= 35:
        return "18-35"
    elif age <= 50:
        return "35-50"
    else:
        return "50+"

users_map = users_data.map(parse_user_age)

users_dict = users_map.collectAsMap()
age_group_dict = {user_id: sperage_age_group(age) for user_id, age in users_dict.items()}


In [43]:
def splition_movie(line):
    parts = line.split(',', 2)
    movie_id = int(parts[0])
    title = parts[1]
    genres = parts[2] if len(parts) > 2 else ""
    return (movie_id, title)

movies_map = movies_data.map(splition_movie)
movies_dict = movies_map.collectAsMap() # Dict to retrieve data


In [44]:
# Split ratings: UserID, MovieID, Rating, Timestamp
def splition_rating_with_user(line):
    parts = line.split(',')
    user_id = int(parts[0])
    movie_id = int(parts[1])
    rating = float(parts[2])
    timestamp = int(parts[3])
    return (user_id, movie_id, rating)

ratings_1_parsed = ratings1_data.map(splition_rating_with_user)
ratings_2_parsed = ratings2_data.map(splition_rating_with_user)

# Combine 2 file
total_ratings = ratings_1_parsed.union(ratings_2_parsed)

In [45]:
# Create dataset (movie_id, (rating, age_group))
def age_group_into_movie_ratings(row):
    user_id, movie_id, rating = row
    age_group = age_group_dict.get(user_id, "Unknown")
    return (movie_id, (rating, age_group))

ratings_into_age_group = total_ratings.map(age_group_into_movie_ratings)

filter_ratings = ratings_into_age_group.filter(lambda x: x[1][1] != "Unknown")

# seperage for each group age
groups_age = ["0-18", "18-35", "35-50", "50+"]
ratings_with_group_age = {}

for group in groups_age:
    ratings_with_group_age[group] = filter_ratings.filter(lambda x: x[1][1] == group).map(lambda x: (x[0], x[1][0]))


In [ ]:

age_averages = {}

# Calculate average for each group
for group in groups_age:
    age_analysis = ratings_with_group_age[group].map(lambda x: (x[0], (x[1], 1))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
    age_averages[group] = age_analysis.map(lambda x: (x[0], x[1][0] / x[1][1]))  # (movie_id, average_rating)

# Calculate result for each film
movie_has_rating = filter_ratings.map(lambda x: x[0]).distinct()

list_movie_has_rating = movie_has_rating.collect()
list_results = []

for movie_id in list_movie_has_rating:
    movie_title = movies_dict.get(movie_id, f"No Name Movie")
    age_ratings = {}

    for group in groups_age:
        movie_ratings = age_averages[group].filter(lambda x: x[0] == movie_id).collect()
        if movie_ratings:
            age_ratings[group] = movie_ratings[0][1]  # average_point
        else:
            age_ratings[group] = None

    list_results.append((movie_id, movie_title, age_ratings))

# Print result
for movie_id, title, age_ratings in list_results:
    ratings_str = ", ".join([
        f"{group}: {age_ratings[group]:.2f}" if age_ratings[group] is not None else f"{group}: NA"
        for group in groups_age
    ])
    print(f"{title} - [{ratings_str}]")